# 27. PyTorch — `torch`
**Python Libraries Lab · Phase 2 · Deep Learning & AI**

_Tensors, automatic gradients and your own training loop._

✅ **Runs in Colab** as-is.

PyTorch is the most popular deep learning library in research and now in industry. Its core object is the tensor, a NumPy style array that can live on a GPU and remember how it was computed. That memory powers autograd, which works out gradients automatically, the numbers that tell each weight which way to move.

**In this notebook:** You will play with tensors and autograd, then build the same digit reader as the TensorFlow lesson as a PyTorch module, train it with your own loop, check its accuracy, and save the weights.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
PyTorch is pre-installed in Colab. A GPU is optional: **Runtime → Change runtime type → T4 GPU**. The code below uses the GPU automatically if there is one.

## Step 1 · Tensors
Like NumPy arrays, plus a `device` and gradient tracking.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2)
print(a + b)
print(a @ b)              # matrix multiply
print(a.mean(), a.shape, a.dtype)
print(torch.from_numpy(__import__("numpy").arange(4)).reshape(2, 2))

## Step 2 · autograd: gradients for free
`requires_grad=True` tracks the maths. `backward()` computes how the output changes with each input.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x            # dy/dx = 2x + 2
y.backward()
print("y =", y.item(), "| dy/dx at x=3:", x.grad.item())   # 8.0

## Step 3 · Load the digits
Same dataset as the TensorFlow lesson, turned into tensors and batches.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import numpy as np, matplotlib.pyplot as plt

digits = load_digits()                      # 1,797 handwritten digits, 8x8 pixels, built into scikit-learn
X = (digits.images / 16.0).astype("float32")  # scale pixels to 0..1
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, label in zip(axes, X_train, y_train):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.show()

from torch.utils.data import TensorDataset, DataLoader
train_dl = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=32, shuffle=True)
X_test_t, y_test_t = torch.tensor(X_test).to(device), torch.tensor(y_test).to(device)

## Step 4 · Define the model
A class with layers in `__init__` and the data flow in `forward`.

In [ ]:
from torch import nn
torch.manual_seed(0)

class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    def forward(self, x):
        return self.net(x)

model = DigitNet().to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Step 5 · The training loop
Predict → loss → backward → step → zero gradients. That is all training is.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def test_accuracy():
    model.eval()
    with torch.no_grad():
        return (model(X_test_t).argmax(1) == y_test_t).float().mean().item()

for epoch in range(1, 31):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    if epoch % 5 == 0:
        print(f"epoch {epoch:>2}  loss {loss.item():.3f}  test accuracy {test_accuracy():.3f}")

## Step 6 · Predict and save
Save only the learned weights (`state_dict`), the standard PyTorch way.

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(X_test_t[:6]), dim=1)
print("Predicted:", probs.argmax(1).tolist(), "| true:", y_test[:6].tolist())

torch.save(model.state_dict(), "digitnet.pt")
again = DigitNet().to(device); again.load_state_dict(torch.load("digitnet.pt"))
print("Reloaded weights OK")

## ✍️ Your turn
Use autograd: for `x = torch.tensor(2.0, requires_grad=True)` and `y = x**3`, compute dy/dx and store it (a Python float) in `answer`. It should be 12.0.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 12.0, f"Expected 12.0, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
y.backward()
answer = x.grad.item()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Tensor**
- **autograd**
- **nn.Module**
- **Optimizer**
- **DataLoader**
- **Device (CPU/GPU)**

**Next:** 28. JAX (`jax`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/28_jax.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install torch scikit-learn matplotlib
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.